# Benchmark de inferência — `winnex-nano`

Braço **winnex-nano** do benchmark pareado. O outro braço é `transformers`, e roda no
notebook irmão com **o mesmo harness** (`bench_core.py`) e a **mesma ordem de
células**. Rode os dois no mesmo tipo de máquina e compare as saídas JSON.

## O que este notebook mede, e em que dtype

| bloco | dtype | por quê |
|---|---|---|
| **Paridade de logits** | **f32** | é o gate. Em f32 os dois motores têm que dar o mesmo argmax em 100% dos tokens gerados; se não derem, nada abaixo é interpretável |
| Perplexidade | **f32** | medida de qualidade — comparar motores, não implementações |
| MMLU / LAMBADA | **f32** | idem; zero-shot, IC de Wilson |
| Prefill / Decode | **bf16** | medida de desempenho, no dtype em que o modelo seria servido |
| Memória | o do bloco | anônimo, RSS e VRAM de pico |

**O dtype está declarado em cada resultado**, no campo `dtype`. Um número sem
ele não é comparável com nada.

## O que este notebook NÃO mede

- **Não é MMLU de 5-shot.** É zero-shot, então os valores absolutos não são
  comparáveis com a literatura.
- **Não mede batch > 1.** Decode é batch 1, que é o caso de um chat.
- **Prefill e decode são regimes diferentes** e têm métricas diferentes:
  prefill é throughput (tokens/s, TTFT), decode é latência (ms/token). Somar
  os dois num único "tokens/s" esconde o que importa.

## Como ler os números de tempo

10 iterações, as 2 primeiras descartadas como warmup (JIT do OpenCL, alocação
preguiçosa, cache frio). Reporta **média, mediana e P99**. O P99 é o que expõe
gargalo de sincronização ou de coleta de lixo: uma mediana boa com P99 ruim é
um serviço que derruba 1 em 100 requisições.


## 1. Instalação

O pacote vem do PyPI. `winnex-madhava` é dependência declarada e é instalado
junto. O motor precisa de um ICD OpenCL; no Kaggle (T4) ele vem com o driver
da NVIDIA.

Se a instalação falhar, a célula abaixo imprime a causa — em particular se o
wheel da versão não existir para este Python/plataforma, o que é diferente de
"a instalação deu erro".


In [ ]:
import subprocess, sys, platform
print("python", platform.python_version(), "|", platform.platform())
VERSAO = "0.1.35"
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"winnex-nano=={VERSAO}"],
                   capture_output=True, text=True)
print("rc =", r.returncode)
if r.returncode != 0:
    print("--- stdout ---"); print(r.stdout[-2500:])
    print("--- stderr ---"); print(r.stderr[-2500:])
    raise SystemExit("instalacao falhou — veja acima qual wheel faltou")
import winnex_nano as wn
print("winnex-nano", wn.__version__)
print("SIMD (AVX2+FMA):", wn.compiled_with_simd())


## 2. O harness

`bench_core.py` é o mesmo arquivo nos dois notebooks. Copiado para cá na
célula seguinte para que o notebook seja autossuficiente no Kaggle.


In [ ]:
import os
CORE = r'''#!/usr/bin/env python3
"""bench_core.py - harness compartilhado pelos dois notebooks de benchmark.

Um unico harness para os dois bracos. Se o transformers e o winnex-nano fossem
medidos por codigos diferentes, a diferenca entre os numeros poderia ser a
diferenca entre os codigos.

Mede:
  PARIDADE (gate f32)  prompt de 50 tokens, argmax de cada token gerado.
  PPL (f32)            exp(Sum nll/Sum tok), wikitext-2, janela 512 stride 256.
  ACURACIA (f32)       MMLU e LAMBADA por log-verossimilhanca, IC de Wilson.
  PREFILL (bf16)       512/1024/2048 tokens: TTFT e tokens/s.
  DECODE (bf16)        prompt 50, 128 tokens, batch 1: ITL ms/token.
  MEMORIA              3 pontos: apos import, apos carga, pico na geracao.
"""
import gc
import math
import os
import statistics
import sys
import time

import numpy as np


def wilson(k, n, z=1.96):
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    d = 1 + z * z / n
    c = p + z * z / (2 * n)
    r = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return ((c - r) / d, (c + r) / d)


def boot_ratio_ci(num, den, n_boot=4000, alpha=0.05, seed=0, exponentiate=True):
    num = np.asarray(num, dtype=np.float64)
    den = np.asarray(den, dtype=np.float64)
    if num.size == 0:
        return (None, None)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, num.size, size=(n_boot, num.size))
    r = num[idx].sum(1) / den[idx].sum(1)
    if exponentiate:
        r = np.exp(r)
    return (float(np.quantile(r, alpha / 2)), float(np.quantile(r, 1 - alpha / 2)))


def pct(values, q):
    v = sorted(values)
    if not v:
        return None
    k = (len(v) - 1) * (q / 100.0)
    f = int(math.floor(k))
    c = min(f + 1, len(v) - 1)
    return v[f] + (v[c] - v[f]) * (k - f)


def describe(times_ms):
    if not times_ms:
        return {}
    return {"n": len(times_ms),
            "mean_ms": round(statistics.fmean(times_ms), 3),
            "median_ms": round(statistics.median(times_ms), 3),
            "p99_ms": round(pct(times_ms, 99), 3),
            "min_ms": round(min(times_ms), 3),
            "max_ms": round(max(times_ms), 3)}


def smaps():
    rss = anon = 0
    try:
        with open("/proc/self/smaps") as f:
            for line in f:
                if line.startswith("Rss:"):
                    rss += int(line.split()[1]) * 1024
                elif line.startswith("Anonymous:"):
                    anon += int(line.split()[1]) * 1024
    except OSError:
        pass
    return rss, anon


def vram():
    try:
        import subprocess
        o = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=10)
        lines = [l for l in o.stdout.strip().splitlines() if l.strip()]
        return sum(int(l) for l in lines) * 2**20
    except Exception:
        return 0


class MemTracker:
    def __init__(self):
        self.peak = 0
        self.base = 0
        self._run = False
        self._t = None

    def __enter__(self):
        import threading
        self.base = vram()
        self.peak = self.base
        self._run = True

        def loop():
            while self._run:
                v = vram()
                if v > self.peak:
                    self.peak = v
                time.sleep(0.05)

        self._t = threading.Thread(target=loop, daemon=True)
        self._t.start()
        return self

    def __exit__(self, *a):
        self._run = False
        if self._t:
            self._t.join(timeout=2)


class WinnexArm:
    name = "winnex-nano"

    def __init__(self, model_dir, device="gpu", dtype="f32"):
        import winnex_nano as wn
        self.wn = wn
        self.model_dir = model_dir
        self.dtype = dtype
        if dtype == "bf16":
            os.environ["WINNEX_GPU_DTYPE"] = "bf16"
        else:
            os.environ.pop("WINNEX_GPU_DTYPE", None)
        self.cfg = wn.load_config(os.path.join(model_dir, "config.json"))
        self.st = wn.Safetensors.open_dir(model_dir)
        self.eng = wn.ForwardEngine(self.cfg, self.st)
        self.device = "cpu"
        if device == "gpu":
            lib = os.environ.get("WINNEX_OPENCL_LIB", "")
            self.eng.enable_gpu(lib)
            self.device = self.eng.weight_mode
        self.V = self.cfg.vocab_size
        self.max_ctx = int(self.eng.max_context)

    def load_tokenizer(self):
        from tokenizers import Tokenizer
        self._tk = Tokenizer.from_file(os.path.join(self.model_dir, "tokenizer.json"))
        return self

    def encode(self, text, add_special_tokens=False):
        return self._tk.encode(text, add_special_tokens=add_special_tokens).ids

    def logits_all(self, ids):
        H = np.ascontiguousarray(self.eng.embed_ids(list(ids)), dtype=np.float32)
        out = self.eng.forward(H, len(ids), True)
        return np.asarray(out, dtype=np.float64).reshape(len(ids), self.V)

    def reset(self):
        self.eng.reset_cache()

    def prefill(self, ids):
        return np.asarray(self.eng.prefill(list(ids)), dtype=np.float64)

    def decode_step(self, token_id):
        return np.asarray(self.eng.step(int(token_id)), dtype=np.float64)

    def sync(self):
        pass


class HFArm:
    name = "transformers"

    def __init__(self, model_dir, device="gpu", dtype="f32"):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.torch = torch
        self.model_dir = model_dir
        self.dtype = dtype
        self.tok = AutoTokenizer.from_pretrained(model_dir)
        td = torch.float32 if dtype == "f32" else torch.bfloat16
        self.dev = "cuda" if (device == "gpu" and torch.cuda.is_available()) else "cpu"
        self.m = AutoModelForCausalLM.from_pretrained(model_dir, dtype=td).to(self.dev).eval()
        self.device = self.dev
        self.V = self.m.config.vocab_size
        self.max_ctx = int(getattr(self.m.config, "max_position_embeddings", 4096))
        self._past = None

    def load_tokenizer(self):
        return self

    def encode(self, text, add_special_tokens=False):
        return self.tok(text, add_special_tokens=add_special_tokens).input_ids

    def logits_all(self, ids):
        t = self.torch
        with t.no_grad():
            o = self.m(t.tensor([list(ids)], device=self.dev))
        return o.logits[0].float().cpu().numpy().astype(np.float64)

    def reset(self):
        self._past = None

    def prefill(self, ids):
        t = self.torch
        with t.no_grad():
            o = self.m(t.tensor([list(ids)], device=self.dev), use_cache=True)
        self._past = o.past_key_values
        return o.logits[0, -1].float().cpu().numpy().astype(np.float64)

    def decode_step(self, token_id):
        t = self.torch
        with t.no_grad():
            o = self.m(t.tensor([[int(token_id)]], device=self.dev),
                       past_key_values=self._past, use_cache=True)
        self._past = o.past_key_values
        return o.logits[0, -1].float().cpu().numpy().astype(np.float64)

    def sync(self):
        if self.dev == "cuda":
            self.torch.cuda.synchronize()


def task_parity(arm, prompt_ids, n_gen=50):
    arm.reset()
    lg = arm.prefill(prompt_ids)
    got = []
    cur = int(np.argmax(lg))
    for _ in range(n_gen):
        got.append(cur)
        lg = arm.decode_step(cur)
        cur = int(np.argmax(lg))
    return got


def task_ppl(arm, seq=512, stride=256, n_windows=40):
    from datasets import load_dataset
    ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
    text = "\n\n".join(r["text"].strip() for r in ds if r["text"].strip())
    ids = arm.encode(text, False)
    wins, i = [], 0
    while len(wins) < n_windows and i + seq <= len(ids):
        wins.append(ids[i:i + seq])
        i += stride
    nlls, counts = [], []
    for w in wins:
        lg = arm.logits_all(w)
        m = lg[:-1].max(1, keepdims=True)
        ls = (lg[:-1] - m) - np.log(np.exp(lg[:-1] - m).sum(1, keepdims=True))
        v = -ls[np.arange(len(w) - 1), np.asarray(w[1:])]
        nlls.append(v)
        counts.append(np.ones_like(v))
    nll = np.concatenate(nlls)
    cnt = np.concatenate(counts)
    lo, hi = boot_ratio_ci(nll, cnt)
    return {"task": "ppl/wikitext-2", "dtype": arm.dtype,
            "ppl": float(math.exp(nll.sum() / cnt.sum())),
            "ppl_ci95": [lo, hi], "n_tokens": int(cnt.sum()),
            "n_janelas": len(wins), "seq": seq, "stride": stride}


def _mcq_ll(arm, items, name, normalize=True):
    hits, n = 0, 0
    for it in items:
        q = it["question"].strip()
        sc = []
        for ch in it["choices"]:
            prompt = arm.encode(q + "\nAnswer:", False)
            cont = arm.encode(" " + ch.strip(), False)
            ids = prompt + cont
            if len(ids) < 2 or len(ids) > arm.max_ctx - 2:
                sc.append(-1e30)
                continue
            lg = arm.logits_all(ids)
            m = lg[:-1].max(1, keepdims=True)
            ls = (lg[:-1] - m) - np.log(np.exp(lg[:-1] - m).sum(1, keepdims=True))
            s = sum(float(ls[len(prompt) - 1 + j, cont[j]]) for j in range(len(cont)))
            sc.append(s / max(1, len(cont)) if normalize else s)
        if not sc:
            continue
        hits += int(int(np.argmax(sc)) == it["answer"])
        n += 1
    lo, hi = wilson(hits, n)
    return {"task": "{}/{}".format(name, "ll-norm" if normalize else "ll-raw"),
            "dtype": arm.dtype, "acc": hits / n if n else None,
            "acc_ci95": [lo, hi], "acertos": hits, "n": n}


def task_mmlu(arm, n=150, seed=0, normalize=True):
    from datasets import load_dataset
    import random
    ds = load_dataset("cais/mmlu", "all", split="test")
    idx = list(range(len(ds)))
    random.Random(seed).shuffle(idx)
    items = [{"question": ds[i]["question"], "choices": list(ds[i]["choices"]),
              "answer": int(ds[i]["answer"])} for i in idx[:n]]
    return _mcq_ll(arm, items, "mmlu", normalize)


def task_lambada(arm, n=150, seed=0):
    from datasets import load_dataset
    import random
    ds = load_dataset("lambada", "plain_text", split="test")
    idx = list(range(len(ds)))
    random.Random(seed).shuffle(idx)
    texts = [ds[i]["text"] for i in idx[:n]]
    hits, nn, skipped = 0, 0, 0
    for t in texts:
        t = t.strip()
        if " " not in t:
            skipped += 1
            continue
        prefix, target = t.rsplit(" ", 1)
        ip = arm.encode(prefix, False)
        it = arm.encode(" " + target, False)
        if not it:
            skipped += 1
            continue
        ids = ip + it
        if len(ids) > arm.max_ctx - 2:
            ids = ids[-(arm.max_ctx - 2):]
            if len(ids) <= len(it):
                skipped += 1
                continue
        lg = arm.logits_all(ids)
        hits += int(int(lg[len(ip) - 1].argmax()) == it[0])
        nn += 1
    lo, hi = wilson(hits, nn)
    return {"task": "lambada", "dtype": arm.dtype,
            "acc": hits / nn if nn else None, "acc_ci95": [lo, hi],
            "acertos": hits, "n": nn, "pulados": skipped}


def _oom(msg):
    return any(k in msg.lower() for k in
               ("out of memory", "oom", "cl_mem", "allocate", "memoryerror"))


def task_prefill(arm, lengths=(512, 1024, 2048), iters=10, warmup=2):
    out = []
    for L in lengths:
        ids = list(range(1000, 1000 + min(L, arm.max_ctx - 8)))
        try:
            for _ in range(warmup):
                arm.reset()
                arm.prefill(ids)
                arm.sync()
            times = []
            for _ in range(iters):
                arm.reset()
                t0 = time.perf_counter()
                arm.prefill(ids)
                arm.sync()
                times.append((time.perf_counter() - t0) * 1e3)
            d = describe(times)
            d.update(task="prefill", dtype=arm.dtype, n_tokens=len(ids),
                     ttft_ms=d["median_ms"],
                     tok_per_s=round(len(ids) / (d["median_ms"] / 1e3), 1))
            out.append(d)
        except (RuntimeError, MemoryError) as e:
            m = str(e)[:140]
            out.append({"task": "prefill", "dtype": arm.dtype, "n_tokens": L,
                        "status": "OOM" if _oom(m) else "ERRO", "erro": m})
            break
    return out


def task_decode(arm, prompt_len=50, n_gen=128, iters=10, warmup=2):
    ids = list(range(1000, 1000 + min(prompt_len, arm.max_ctx - n_gen - 8)))
    try:
        for _ in range(warmup):
            arm.reset()
            lg = arm.prefill(ids)
            cur = int(np.argmax(lg))
            for _ in range(16):
                cur = int(np.argmax(arm.decode_step(cur)))
            arm.sync()
        runs = []
        for _ in range(iters):
            arm.reset()
            lg = arm.prefill(ids)
            arm.sync()
            cur = int(np.argmax(lg))
            t0 = time.perf_counter()
            for _ in range(n_gen):
                lg = arm.decode_step(cur)
                cur = int(np.argmax(lg))
            arm.sync()
            runs.append((time.perf_counter() - t0) * 1e3 / n_gen)
        d = describe(runs)
        d.update(task="decode", dtype=arm.dtype, prompt_len=len(ids), n_gen=n_gen,
                 batch=1, itl_ms=d["median_ms"],
                 tok_per_s=round(1000.0 / d["median_ms"], 2) if d["median_ms"] else None)
        return d
    except (RuntimeError, MemoryError) as e:
        m = str(e)[:140]
        return {"task": "decode", "dtype": arm.dtype,
                "status": "OOM" if _oom(m) else "ERRO", "erro": m}
'''
with open("bench_core.py", "w") as f:
    f.write(CORE)
import bench_core
print("harness carregado;", len(CORE), "bytes")


## 3. Memória: linha de base (ponto 1 de 3)

Medida **antes** de carregar o modelo. Heap anônimo é a métrica honesta para o
winnex, que lê o checkpoint por mmap: as páginas de arquivo aparecem no RSS sem
serem memória alocada pelo processo.


In [ ]:
import gc
gc.collect()
rss0, anon0 = bench_core.smaps()
vram0 = bench_core.vram()
BASELINE = {"rss_gb": rss0/2**30, "anon_gb": anon0/2**30, "vram_gb": vram0/2**30}
print(f"baseline  RSS={BASELINE['rss_gb']:.3f} GB  "
      f"anonimo={BASELINE['anon_gb']:.3f} GB  VRAM={BASELINE['vram_gb']:.3f} GB")


## 4. Modelo

Qwen2.5-0.5B-Instruct: pequeno o bastante para caber na T4 em f32 (1,0 GB de
pesos), o que permite rodar o gate de paridade e a PPL **em f32** — condição
para comparar motores sem a confusão do bf16.

Trocar o `MODELO` para um Llama-3.2 exige que o winnex-nano honre
`rope_scaling` (corrigido na 0.1.35); numa versão anterior ele rodaria errado
sem avisar.


In [ ]:
import os
MODELO_ID = "Qwen/Qwen2.5-0.5B-Instruct"
MODELO = "/kaggle/working/modelo"
if not os.path.isdir(MODELO):
    from huggingface_hub import snapshot_download
    snapshot_download(MODELO_ID, local_dir=MODELO,
                      allow_patterns=["*.json", "*.safetensors", "*.txt", "tokenizer*"])
print(sorted(os.listdir(MODELO)))


## 5. Carga do modelo (ponto 2 de 3 de memória)

`WinnexArm`, dispositivo `gpu`, dtype `f32` — o gate e a qualidade precisam de f32.
Se a GPU não estiver disponível o braço cai para CPU e **declara isso** em
`arm.device`.


In [ ]:
import time, importlib
importlib.reload(bench_core)

t0 = time.perf_counter()
arm = bench_core.WinnexArm(MODELO, device="gpu", dtype="f32").load_tokenizer()
t_load = time.perf_counter() - t0

rss1, anon1 = bench_core.smaps()
CARGA = {"load_s": round(t_load, 2),
         "anon_delta_gb": round((anon1-anon0)/2**30, 3),
         "vram_delta_gb": round((bench_core.vram()-vram0)/2**30, 3)}
print(f"carregou em {t_load:.2f} s | modo={arm.device}")
print(f"anonimo +{CARGA['anon_delta_gb']:.3f} GB | VRAM +{CARGA['vram_delta_gb']:.3f} GB")
print("vocab", arm.V, "| contexto maximo", arm.max_ctx)


## 6. GATE: paridade de logits (f32)

Prompt de 50 tokens, 50 tokens gerados, argmax de cada um comparado com a
referência publicada pelo braço `transformers`.

**Se este gate falhar, nada abaixo vale.** É por isso que ele vem antes da
perplexidade e das tarefas: medir qualidade num motor que não reproduz o outro
mede o motor, não o modelo.

A referência é gravada em `referencia_paridade.json` pelo notebook do
`transformers`; rode os dois e copie o arquivo entre eles (ou compare os dois JSON
no fim).


In [ ]:
PROMPT_TEXTO = ("The history of the Roman Empire is a long and complicated "
                "story that begins with the founding of the city")
prompt_ids = arm.encode(PROMPT_TEXTO, False)[:50]
print("prompt:", len(prompt_ids), "tokens")

t0 = time.perf_counter()
gerado = bench_core.task_parity(arm, prompt_ids, n_gen=50)
t_par = time.perf_counter() - t0
print("50 tokens gerados em", round(t_par, 2), "s")
print("tokens:", gerado[:20], "...")

import json, os
REF = "referencia_paridade.json"
if os.path.exists(REF):
    ref = json.load(open(REF))
    igual = sum(a == b for a, b in zip(gerado, ref["tokens"]))
    print(f"\nGATE: {igual}/{len(ref['tokens'])} argmax identicos "
          f"({100*igual/len(ref['tokens']):.1f}%)")
    assert igual == len(ref["tokens"]), "GATE FALHOU — nao siga para as metricas"
    print("GATE PASSOU")
else:
    json.dump({"arm": "winnex-nano", "dtype": "f32", "prompt_ids": prompt_ids,
               "tokens": gerado}, open(REF, "w"), indent=1)
    print("referencia gravada em", REF, "— rode o outro notebook e volte")


## 7. Perplexidade de corpus (f32)

Wikitext-2 test, janelas de 512 com passo 256. `PPL = exp(Σnll/Σtok)`.

O IC é bootstrap sobre **tokens**, não sobre janelas: as janelas se sobrepõem
em 50% e compartilham metade dos tokens, então tratá-las como amostras
independentes produz um intervalo que não se sustenta.


In [ ]:
t0 = time.perf_counter()
r_ppl = bench_core.task_ppl(arm, seq=512, stride=256, n_windows=40)
r_ppl["segundos"] = round(time.perf_counter()-t0, 1)
print(f"PPL = {r_ppl['ppl']:.4f}  IC95 = [{r_ppl['ppl_ci95'][0]:.4f}, {r_ppl['ppl_ci95'][1]:.4f}]")
print(f"n = {r_ppl['n_tokens']} tokens, {r_ppl['n_janelas']} janelas, "
      f"seq={r_ppl['seq']} stride={r_ppl['stride']}, {r_ppl['segundos']}s")
print("dtype:", r_ppl["dtype"])


## 8. Acurácia zero-shot (f32)

MMLU e LAMBADA por **log-verossimilhança**, não por geração: pontua cada
alternativa pela soma (ou média, na variante `ll-norm`) dos log-probs da
continuação e escolhe a maior. É o protocolo do MMLU original.

IC de Wilson. Com n=150 o intervalo de uma acurácia perto de 35% tem ~±8
pontos — diferenças menores que isso não são detectáveis e não devem ser
reportadas como vantagem.

`n` é ajustável; cada item de MMLU custa 4 forwards completos.


In [ ]:
N_ITENS = 150
t0 = time.perf_counter()
r_mmlu_raw  = bench_core.task_mmlu(arm, n=N_ITENS, normalize=False)
r_mmlu_norm = bench_core.task_mmlu(arm, n=N_ITENS, normalize=True)
r_lambada   = bench_core.task_lambada(arm, n=N_ITENS)
print("tempo total:", round(time.perf_counter()-t0, 1), "s")
for r in (r_mmlu_raw, r_mmlu_norm, r_lambada):
    lo, hi = r["acc_ci95"]
    print(f"  {r['task']:<16} {r['acc']*100:5.1f}%  n={r['n']:<4} "
          f"IC95=[{lo*100:.1f}, {hi*100:.1f}]  ({r['dtype']})")
print("\nNota: zero-shot. O MMLU da literatura e 5-shot — nao compare o valor absoluto.")


## 9. Prefill — throughput (bf16)

Recarrega o modelo em **bf16**, que é o dtype em que ele seria servido. O
dtype é declarado no resultado.

512 / 1024 / 2048 tokens. Métrica: **TTFT** (tempo até o primeiro token) e
**tokens/s**. 10 iterações, 2 descartadas.

Prefill e decode são regimes de memória e computação diferentes. Misturá-los
num único "tokens/s" esconde o gargalo: aqui há paralelismo sobre a sequência;
no decode há um token por vez.


In [ ]:
%time
del arm
gc.collect()
arm16 = bench_core.WinnexArm(MODELO, device="gpu", dtype="bf16").load_tokenizer()
print("modo:", arm16.device, "| dtype:", arm16.dtype)

r_prefill = bench_core.task_prefill(arm16, lengths=(512, 1024, 2048), iters=10, warmup=2)
for r in r_prefill:
    if r.get("status"):
        print(f"  {r['n_tokens']:>5} tokens: {r['status']} — {r.get('erro','')[:90]}")
    else:
        print(f"  {r['n_tokens']:>5} tokens: TTFT {r['ttft_ms']:8.1f} ms  "
              f"{r['tok_per_s']:8.1f} tok/s  "
              f"(mediana {r['median_ms']:.1f}, P99 {r['p99_ms']:.1f})")


## 10. Decode — latência (bf16)

Prompt de 50 tokens, 128 tokens gerados, **batch = 1**. Métrica:
**ITL** (inter-token latency, ms/token) e tokens/s.

O P99 importa mais que a média: é o que determina se um timeout de 200 ms
derruba uma requisição em 100. Uma mediana boa com P99 ruim é um serviço
instável, mesmo com throughput médio bom.


In [ ]:
r_decode = bench_core.task_decode(arm16, prompt_len=50, n_gen=128, iters=10, warmup=2)
if r_decode.get("status"):
    print(r_decode["status"], "-", r_decode.get("erro","")[:120])
else:
    print(f"ITL mediana = {r_decode['itl_ms']:.2f} ms/token  "
          f"({r_decode['tok_per_s']:.2f} tok/s)")
    print(f"  media  {r_decode['mean_ms']:.2f} ms")
    print(f"  P99    {r_decode['p99_ms']:.2f} ms   <- expos gargalo de sync/GC")
    print(f"  min/max {r_decode['min_ms']:.2f} / {r_decode['max_ms']:.2f} ms")
    print(f"  n = {r_decode['n']} execucoes x {r_decode['n_gen']} tokens, dtype {r_decode['dtype']}")


## 11. Memória — pico durante a geração (ponto 3 de 3)

- **Heap anônimo**: a RAM realmente alocada pelo processo. Ignora páginas de
  arquivo do mmap, que o SO pode descartar. É a métrica honesta para o winnex.
- **RSS**: memória residente total; mostra o impacto do mmap no sistema.
- **VRAM pico**: amostrada em thread durante a geração, incluindo KV cache.


In [ ]:
with bench_core.MemTracker() as mt:
    pb = list(range(1000, 1000 + min(512, arm16.max_ctx - 40)))
    arm16.reset(); lg = arm16.prefill(pb)
    cur = int(lg.argmax())
    for _ in range(64):
        lg = arm16.decode_step(cur); cur = int(lg.argmax())
    arm16.sync()

rss2, anon2 = bench_core.smaps()
PICO = {"anon_gb": anon2/2**30,
        "rss_gb": rss2/2**30,
        "vram_peak_gb": (mt.peak - BASELINE["vram_gb"]*2**30)/2**30}
print(f"heap anonimo    {PICO['anon_gb']:7.3f} GB   (baseline {BASELINE['anon_gb']:.3f})")
print(f"RSS             {PICO['rss_gb']:7.3f} GB")
print(f"VRAM pico       {PICO['vram_peak_gb']:7.3f} GB")
print(f"  -> anonimo e a metrica honesta: o mmap do checkpoint aparece no RSS")
print(f"     sem ser memoria alocada pelo processo.")


## 12. Limpeza e vazamento

Libera o motor, força coleta e mede o heap anônimo de novo. O delta depois da
limpeza é o vazamento. **Deve ser ~0**: memória que sobrevive ao `del` é
memória que um servidor não recupera entre requisições.


In [ ]:
del arm16
gc.collect()
if torch_ok := ("torch" in sys.modules):
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
time.sleep(1)
rss3, anon3 = bench_core.smaps()
VAZAMENTO = {"anon_gb": (anon3-anon0)/2**30,
             "vram_gb": (bench_core.vram()-vram0)/2**30}
print(f"apos limpeza: anonimo {anon3/2**30:.3f} GB  (baseline {anon0/2**30:.3f})")
print(f"vazamento anonimo: {VAZAMENTO['anon_gb']:+.3f} GB")
print(f"vazamento VRAM   : {VAZAMENTO['vram_gb']:+.3f} GB")


## 13. Exportar

Um JSON por braço, com **todos** os números e o dtype de cada um. É esse
arquivo que a comparação lê — não a saída impressa, que não carrega n nem IC.


In [ ]:
import json, platform
saida = {
    "arm": "winnex-nano",
    "maquina": platform.platform(),
    "python": platform.python_version(),
    "modelo": MODELO_ID,
    "data": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "memoria": {"baseline": BASELINE, "carga": CARGA, "pico": PICO, "vazamento": VAZAMENTO},
    "gate_paridade": {"dtype": "f32", "n_tokens": len(gerado),
                       "primeiros": gerado[:10]},
    "resultados": [r_ppl, r_mmlu_raw, r_mmlu_norm, r_lambada,
                   *r_prefill, r_decode],
}
ARQ = "resultado_winnex_nano.json"
json.dump(saida, open(ARQ, "w"), indent=2, default=float)
print("gravado:", ARQ)
print(json.dumps({k: v for k, v in saida.items() if k != "resultados"}, indent=2, default=float)[:900])
